# 09H · Reuse the original 09B region

This notebook compares SPARKLE and SpotClean across the **saved 09B evaluation disk**. Its section, center, radius and original evaluation cells come from the completed 09B run. A regular grid covers the disk at native 2 µm resolution. Marker abundance, immune-cell counts and the percentage of low-H&E squares place **no threshold on region selection**.

Run this notebook beside 09G, in the same CT26 SPARKLE spatial CPU kernel. Choose **Run All** to process the full disk. To time the first two tiles, set `TILES_PER_RUN = 2`. Then set it back to `None` and rerun with the same settings. Each completed method fit has a checkpoint, so the next invocation can reuse it.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

from bins09d import STUDY_ROOT, TEMP_PROJECT_ROOT
from review09 import show_review_zip
from run09h import run_09h

RESULT_ROOT = Path("09_pilot_results")
RUN_DIR = None          # Or the directory containing the original 09B complete.json
PATH_OVERRIDES = {}     # Same source keys as 09F/09G, e.g. "zarr" and "aligned_qc"
RSCRIPT = None          # Reuse the matching 09F R executable if available
ZARR_TABLE = "square_002um"

TILES_PER_RUN = None    # Full target; use 2 for a timed, resumable first pass
RUN_TAG = ""            # Keep this value to resume; choose a new tag for a fresh run
SIDE_BINS, HALO_BINS = 64, 6
SCORE_MAX = 0.10
REQUIRE_SR_BACKGROUND = False
SPARKLE_BIN_UM, SPARKLE_RADIUS_UM = 12., 50.
SPOTCLEAN_MINUTES = 60  # Wall-time limit for each tile's R fit, excluding setup

## Fit the disk in bounded windows

At the defaults, each model sees a **128 × 128 µm** fit window. Neighboring windows overlap, with a nominal 12 µm margin on each side of the 104 µm ownership grid. Each observed square inside the disk has a single owner for the raw RNA readout. Each original cell is evaluated once when its owner has the complete Proseg footprint, including measured squares that cross the ownership boundary.

Both methods receive the same raw input and assignments in each window. SPARKLE estimates leakage using out-of-mask counts. SpotClean treats unassigned squares with a finite stored `he_tissue_score` ≤ `SCORE_MAX` as provisional background, including candidates within the Space Ranger tissue boundary by default. A tile with insufficient input for one method can still contribute raw data and a fit from the other method. The local fits have finite spatial context, especially near window edges.

In [ ]:
out, archive, status = run_09h(
    result_root=RESULT_ROOT, run_dir=RUN_DIR, study_root=STUDY_ROOT,
    temp_project_root=TEMP_PROJECT_ROOT, overrides=PATH_OVERRIDES,
    zarr_table=ZARR_TABLE, rscript=RSCRIPT, side_bins=SIDE_BINS, halo_bins=HALO_BINS,
    score_max=SCORE_MAX, require_sr_background=REQUIRE_SR_BACKGROUND,
    sparkle_bin_um=SPARKLE_BIN_UM, sparkle_radius_um=SPARKLE_RADIUS_UM,
    fit_minutes=SPOTCLEAN_MINUTES, max_new_tiles=TILES_PER_RUN, run_tag=RUN_TAG,
)
print("Run status", status["state"])
for key in ("reason", "r_setup_error"):
    if status.get(key):
        print(status[key])
review_download = show_review_zip(archive)

## Check the area, cells and runtime

The saved geometry identifies the region used. Compare the reviewed and paired rows to see how much of the disk supports both fits. The map shows the staining score and model availability at each prepared disk position. Timing summarizes successful SpotClean processes, including any reused fits. Early tiles favor locations where both fits have sufficient input. They can overrepresent well-supported fits. Full-disk runtime for this specimen is unknown.

In [ ]:
regions = out / "regions"
comparison = out / "comparison"
readout_current = status.get("verified_pairs") is not None
receipt = regions / "coverage.json"
if "total_tiles" in status and receipt.is_file():
    coverage = json.loads(receipt.read_text())
    geometry = coverage["geometry"]
    center = tuple(round(x, 1) for x in geometry["center_um"])
    print(f'{coverage["sample"]}  |  center {center} µm  |  radius {geometry["inner_radius_um"]:.1f} µm')
    print(f'{coverage["n_tiles_prepared"]:,} of {coverage["n_tiles_planned"]:,} tiles prepared, '
          f'{coverage["n_disk_bins_prepared"]:,} of {coverage["n_disk_bins_reference"]:,} measured disk squares.')
    reasons = coverage.get("prior_cell_counts_by_reason", {})
    if reasons:
        display(pd.Series(reasons, name="09B cells").rename_axis("footprint status").to_frame())
if readout_current and (comparison / "domain_coverage.csv").is_file():
    display(pd.read_csv(comparison / "domain_coverage.csv").round(3))

progress_path = out / "progress.csv"
if "total_tiles" in status and progress_path.is_file() and progress_path.stat().st_size > 1:
    progress = pd.read_csv(progress_path)
    states = pd.concat([progress[m].value_counts().rename(m) for m in ("sparkle", "spotclean")], axis=1)
    display(states.fillna(0).astype(int).rename_axis("fit status"))
    if "spotclean_seconds" in progress:
        seconds = pd.to_numeric(progress["spotclean_seconds"], errors="coerce").dropna()
        if len(seconds):
            print(f'SpotClean R wall time over {len(seconds)} completed fits: '
                  f'{seconds.sum()/60:.1f} min total, {seconds.median():.1f} s median, '
                  f'{seconds.max():.1f} s longest.')
    if "spotclean_peak_R_MiB" in progress:
        peak = pd.to_numeric(progress["spotclean_peak_R_MiB"], errors="coerce").max()
        if pd.notna(peak):
            print(f'Peak measured R memory across completed fits: {peak/1024:.2f} GiB.')
if readout_current and (out / "disk_footprint.png").is_file():
    display(Image(filename=str(out / "disk_footprint.png"), width=900))

## Read the paired marker counts

Read myeloid counts in nearby pan-T-supported cells alongside their retention in macrophage/monocyte-labeled cells. In each row, both methods use **the same genes, complete cells and raw UMI denominator**. A low denominator gives a fragile percentage. Missing estimates remain blank. Values above 100% are possible with SpotClean. An unchanged SPARKLE count can reflect a gene that did not pass the local subtraction threshold.

The background table repeats the paired comparison separately for tiles below and at or above 25% provisional SpotClean background. The cutoff describes the local fit conditions. Estimates from sparse-background tiles depend more heavily on which squares supply that background. Each category can cover different parts of this one section. Counts for Cd4, Cd8a and Cd8b1 inside and outside Proseg masks by staining score are in `comparison/coreceptors_disk_by_score.csv`.

In [ ]:
paired_path = comparison / "pooled_retention.csv"
if readout_current and paired_path.is_file():
    paired = pd.read_csv(paired_path)
    if paired.empty:
        print("No marker group has a verified paired estimate in this run.")
    else:
        display(paired.drop(columns="region_names").round(3))
        display(Image(filename=str(comparison / "paired_retention.png"), width=650))
sensitivity_path = comparison / "background_sensitivity.csv"
if readout_current and sensitivity_path.is_file():
    bands = pd.read_csv(sensitivity_path)
    display(bands[["background_fraction_band", "prepared_tiles", "group", "panel", "paired_regions",
                   "cells", "raw_UMIs", "sparkle_retention", "spotclean_retention"]].round(3))

## Return the results

The link above downloads `09H_return_to_chat_*.zip`. It contains the pooled tables, figure, coverage and per-tile status. A partial or diagnostic ZIP can also be returned. The local results directory retains the full fits and native matrices. Inspect `regions/local/cell_coverage.csv.gz` for each original cell’s coverage and `progress.csv` for each tile’s fit status.

In [ ]:
print("Results directory", out)
print("Return ZIP", review_download)